In [0]:
%sql
USE CATALOG assignment1;
USE SCHEMA silver;

In [0]:
%sql
-- REGION
DROP TABLE IF EXISTS region;
CREATE TABLE region (
  region_key   INT            NOT NULL,
  name         STRING         NOT NULL,
  comment      STRING,
  CONSTRAINT pk_region PRIMARY KEY (region_key)
) USING DELTA;

INSERT INTO region
SELECT r_regionkey, r_name, r_comment
FROM assignment1.bronze.region;

-- NATION
DROP TABLE IF EXISTS nation;
CREATE TABLE nation (
  nation_key   INT            NOT NULL,
  name         STRING         NOT NULL,
  region_key   INT            NOT NULL,
  comment      STRING,
  CONSTRAINT pk_nation PRIMARY KEY (nation_key),
  CONSTRAINT fk_nation_region FOREIGN KEY (region_key) REFERENCES region(region_key)
) USING DELTA;

INSERT INTO nation
SELECT n_nationkey, n_name, n_regionkey, n_comment
FROM assignment1.bronze.nation;


In [0]:
%sql
-- SUPPLIER
DROP TABLE IF EXISTS supplier;
CREATE TABLE supplier (
  supplier_key INT            NOT NULL,
  name         STRING         NOT NULL,
  address      STRING,
  nation_key   INT            NOT NULL,
  phone        STRING,
  acctbal      DECIMAL(15,2),
  comment      STRING,
  CONSTRAINT pk_supplier PRIMARY KEY (supplier_key),
  CONSTRAINT fk_supplier_nation FOREIGN KEY (nation_key) REFERENCES nation(nation_key)
) USING DELTA;

INSERT INTO supplier
SELECT s_suppkey, s_name, s_address, s_nationkey, s_phone, s_acctbal, s_comment
FROM assignment1.bronze.supplier;

-- CUSTOMER
DROP TABLE IF EXISTS customer;
CREATE TABLE customer (
  customer_key INT            NOT NULL,
  name         STRING         NOT NULL,
  address      STRING,
  nation_key   INT            NOT NULL,
  phone        STRING,
  acctbal      DECIMAL(15,2),
  mktsegment   STRING,
  comment      STRING,
  CONSTRAINT pk_customer PRIMARY KEY (customer_key),
  CONSTRAINT fk_customer_nation FOREIGN KEY (nation_key) REFERENCES nation(nation_key)
) USING DELTA;

INSERT INTO customer
SELECT c_custkey, c_name, c_address, c_nationkey, c_phone, c_acctbal, c_mktsegment, c_comment
FROM assignment1.bronze.customer;

-- PART
DROP TABLE IF EXISTS part;
CREATE TABLE part (
  part_key     INT            NOT NULL,
  name         STRING         NOT NULL,
  mfgr         STRING,
  brand        STRING,
  type         STRING,
  size         INT,
  container    STRING,
  retailprice  DECIMAL(15,2),
  comment      STRING,
  CONSTRAINT pk_part PRIMARY KEY (part_key)
) USING DELTA;

INSERT INTO part
SELECT p_partkey, p_name, p_mfgr, p_brand, p_type, p_size, p_container, p_retailprice, p_comment
FROM assignment1.bronze.part;

In [0]:
%sql
DROP TABLE IF EXISTS partsupp;
CREATE TABLE partsupp (
  part_key     INT            NOT NULL,
  supplier_key INT            NOT NULL,
  availqty     INT,
  supplycost   DECIMAL(15,2),
  comment      STRING,
  CONSTRAINT pk_partsupp PRIMARY KEY (part_key, supplier_key),
  CONSTRAINT fk_ps_part     FOREIGN KEY (part_key)     REFERENCES part(part_key),
  CONSTRAINT fk_ps_supplier FOREIGN KEY (supplier_key) REFERENCES supplier(supplier_key)
) USING DELTA;

INSERT INTO partsupp
SELECT ps_partkey, ps_suppkey, ps_availqty, ps_supplycost, ps_comment
FROM assignment1.bronze.partsupp;


In [0]:
%sql
DROP TABLE IF EXISTS orders;
CREATE TABLE orders (
  order_key     INT            NOT NULL,
  customer_key  INT            NOT NULL,
  orderstatus   STRING,
  totalprice    DECIMAL(15,2),
  orderdate     DATE,
  orderpriority STRING,
  clerk         STRING,
  shippriority  INT,
  comment       STRING,
  CONSTRAINT pk_orders PRIMARY KEY (order_key),
  CONSTRAINT fk_orders_customer FOREIGN KEY (customer_key) REFERENCES customer(customer_key)
) USING DELTA;

INSERT INTO orders
SELECT o_orderkey, o_custkey, o_orderstatus, o_totalprice, o_orderdate,
       o_orderpriority, o_clerk, o_shippriority, o_comment
FROM assignment1.bronze.orders;

In [0]:
%sql
DROP TABLE IF EXISTS lineitem;

CREATE TABLE lineitem (
  order_key     INT           NOT NULL,
  line_number   INT           NOT NULL,
  part_key      INT           NOT NULL,
  supplier_key  INT           NOT NULL,
  quantity      DECIMAL(15,2) NOT NULL,
  extendedprice DECIMAL(15,2),
  discount      DECIMAL(15,2),
  tax           DECIMAL(15,2),
  returnflag    STRING,
  linestatus    STRING,
  shipdate      DATE,
  commitdate    DATE,
  receiptdate   DATE,
  shipinstruct  STRING,
  shipmode      STRING,
  comment       STRING,

  -- constraints
  CONSTRAINT pk_lineitem        PRIMARY KEY (order_key, line_number),
  -- optional if your data uses fractions 0..1
  -- CONSTRAINT ck_discount_rng  CHECK (discount BETWEEN 0 AND 1),
  -- CONSTRAINT ck_tax_rng       CHECK (tax BETWEEN 0 AND 1),

  CONSTRAINT fk_li_order        FOREIGN KEY (order_key)    REFERENCES orders(order_key),
  CONSTRAINT fk_li_part         FOREIGN KEY (part_key)     REFERENCES part(part_key),
  CONSTRAINT fk_li_supplier     FOREIGN KEY (supplier_key) REFERENCES supplier(supplier_key),
  CONSTRAINT fk_li_partsupp     FOREIGN KEY (part_key, supplier_key)
                                REFERENCES partsupp(part_key, supplier_key)
) USING DELTA;

INSERT INTO lineitem
SELECT l_orderkey,
       l_linenumber,
       l_partkey,
       l_suppkey,
       l_quantity,
       l_extendedprice,
       l_discount,
       l_tax,
       l_returnflag,
       l_linestatus,
       l_shipdate,
       l_commitdate,
       l_receiptdate,
       l_shipinstruct,
       l_shipmode,
       l_comment
FROM assignment1.bronze.lineitem;
